# Phase 01: Project Overview - Wissenschaftliche Grundlagen und Versuchsaufbau
## Kurs: Machine Learning in der biomedizinischen Forschung (Maus-Monitor-ML)

--- 

## 1. Einführung

Willkommen zum ersten Modul dieses umfassenden Machine-Learning-Projekts. In diesem Projekt begleite ich Sie als Mentor Schritt für Schritt durch den gesamten Prozess eines professionellen Data-Science-Projekts. Unser Ziel ist es nicht, schnell ein Standardmodell zu trainieren. Vielmehr wollen wir die biologischen, mathematischen und statistischen Prinzipien hinter jeder Zeile Code tiefgründig verstehen und sauber dokumentieren.

### Der biologische Kontext: Das DSS-Modell
In der biomedizinischen Forschung ist die Untersuchung von chronisch-entzündlichen Darmerkrankungen (wie Colitis ulcerosa oder Morbus Crohn) von zentraler Bedeutung. Um therapeutische Ansätze zu testen und die Pathophysiologie zu verstehen, wird häufig das **DSS-induzierte Colitis-Modell** bei Mäusen verwendet. 

**Dextransulfat-Natrium (DSS)** ist ein sulfatiertes Polysaccharid, das Mäusen über das Trinkwasser verabreicht wird. Es wirkt toxisch auf das Darmepithel, zerstört die Schleimhautbarriere und induziert eine akute, dosisabhängige Entzündung des Dickdarms, die der humanen Colitis sehr ähnelt.

### Die gemessenen Variablen
Über einen Zeitraum von 14 Tagen (Tag 0 bis Tag 13) wurden täglich folgende Parameter erhoben:
- **`id`**: Eindeutiger Identifikator des einzelnen Tieres (z. B. `SvBi029`). Dies ist eine kategoriale Variable (Nominalskala).
- **`DSS`**: Die Dosisgruppe des Dextransulfat-Natriums (kategoriale Variable auf Ordinalskala):
  - `0`: Kontrollgruppe (0% DSS) - keine Entzündungsinduktion.
  - `1`: Niedrige Dosis (1.0% DSS) - moderate Entzündungsinduktion.
  - `2`: Hohe Dosis (1.5% DSS) - schwere Entzündungsinduktion.
- **`day`**: Der Versuchstag als diskrete zeitliche Variable (Verhältnisskala, $t \in \{0, 1, \dots, 13\}$).
- **`bwc`** (*Body Weight Change*): Die prozentuale Gewichtsentwicklung bezogen auf das Ausgangsgewicht an Tag 0 (Metrische Verhältnisskala).
- **`vwr`** (*Voluntary Wheel Running*): Die freiwillige Laufradaktivität des Tieres, gemessen in Radumdrehungen pro Minute (*revolutions per minute*, rpm) über die Nachtphase (Metrische Verhältnisskala).

### Die humane Endpunkt-Problematik (Missing Values)
Aus ethischen Gründen (Tierschutzgesetze und *3R-Prinzip*: *Replacement, Reduction, Refinement*) dürfen Tiere im Versuch nicht unbegrenzt leiden. Überschreitet der Gewichtsverlust oder der klinische Zustand eines Tieres vordefinierte Abbruchkriterien (sogenannte humane Endpunkte), muss das Tier vorzeitig euthanasiert werden. Dies führt dazu, dass im Datensatz ab einem bestimmten Tag für diese Tiere keine Messwerte mehr vorliegen. Im Machine Learning betrachten wir dies als **strukturell fehlende Werte** (*Missing Not At Random, MNAR*), da das Fehlen der Daten direkt mit der Schwere der Erkrankung (und somit dem Zielwert) korreliert.

--- 

## 2. Lernziele

Am Ende dieser Phase sollten Sie Folgendes verstanden haben:
1. **Biologische Domäne**: Warum und wie das DSS-Modell eingesetzt wird.
2. **Datenstruktur**: Welche Skalenniveaus die Variablen besitzen und warum fehlende Werte entstehen.
3. **Wissenschaftliche Methodik**: Die Phasenstruktur dieses Projekts (30 Module) und das Prinzip der reproduzierbaren Forschung.
4. **Git-Workflow**: Den professionellen Branch-, Commit- und Push-Zyklus für Machine-Learning-Repositoren.

--- 

## 3. Theorie & Intuition

### Warum messen wir BWC und VWR?
- **Gewichtsverlust (BWC)** ist ein klassischer, systemischer Krankheitsindikator. Eine Entzündung führt zu Flüssigkeitsverlust, reduzierter Nahrungsaufnahme und Muskelatrophie. Das Gewicht reagiert jedoch relativ träge.
- **Laufradaktivität (VWR)** ist ein sensitiver Verhaltensparameter. Entzündungsschmerz, Lethargie und Fieber führen sofort zu einer Reduktion des Bewegungsdrangs. Die Intuition besagt: Verhaltensänderungen treten zeitlich *vor* schweren systemischen Veränderungen (Gewichtsverlust) auf. VWR könnte somit ein Frühwarnindikator für Belastung sein.

### Das 30-Phasen-Konzept
Unser Projekt folgt dem CRISP-DM-Modell (*Cross-Industry Standard Process for Data Mining*), erweitert um moderne Machine-Learning-Best-Practices (wie Modell-Erklärbarkeit mit SHAP und rigorose statistische Validierung). Jedes Notebook baut logisch auf dem vorherigen auf.

--- 

## 4. Mathematische Grundlagen

### Formel der Gewichtsentwicklung (BWC)
Die Variable `bwc` ist relativ zum Ausgangsgewicht an Tag 0 definiert. Sei $W_{i,t}$ das absolute Gewicht der Maus $i$ am Tag $t$. Die prozentuale Gewichtsentwicklung $BWC_{i,t}$ berechnet sich wie folgt:

$$BWC_{i,t} = \frac{W_{i,t}}{W_{i,0}} \times 100$$

**Mathematische Eigenschaften:**
- Für $t = 0$ gilt definitionsgemäß: $BWC_{i,0} = \frac{W_{i,0}}{W_{i,0}} \times 100 = 100.0\%$.
- Ein Wert $BWC_{i,t} < 100$ impliziert einen Gewichtsverlust (z. B. $92.5\%$ bedeutet $7.5\%$ Gewichtsverlust gegenüber dem Start).
- Ein Wert $BWC_{i,t} > 100$ impliziert eine Gewichtszunahme.

**Statistische Begründung der Normierung:**
Durch die Division durch $W_{i,0}$ eliminieren wir den Einfluss der biologischen Variabilität des absoluten Startgewichts. Eine Maus, die mit $20\,\text{g}$ startet, verhält sich bezüglich absoluten Gewichtsverlusts anders als eine Maus mit $26\,\text{g}$. Die Normierung macht die Daten zwischen den Tieren vergleichbar.

--- 

## 5. Python-Umsetzung

Wir prüfen zunächst unsere Systemumgebung und die installierten Bibliotheken, um die Reproduzierbarkeit unseres Projekts sicherzustellen. Dies entspricht professionellen Software-Engineering-Standards im Data-Science-Bereich.

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib
import seaborn as sns
import sklearn
import scipy

print("--- Systemumgebung ---")
print(f"Python Version:      {sys.version}")
print(f"Pandas Version:      {pd.__version__}")
print(f"Numpy Version:       {np.__version__}")
print(f"Matplotlib Version:  {matplotlib.__version__}")
print(f"Seaborn Version:     {sns.__version__}")
print(f"Scikit-Learn Version:{sklearn.__version__}")
print(f"Scipy Version:       {scipy.__version__}")

--- Systemumgebung ---
Python Version:      3.12.2 | packaged by conda-forge | (main, Feb 16 2024, 20:54:21) [Clang 16.0.6 ]
Pandas Version:      2.3.3
Numpy Version:       1.26.4
Matplotlib Version:  3.10.8
Seaborn Version:     0.13.2
Scikit-Learn Version:1.8.0
Scipy Version:       1.17.0


--- 

## 6. Visualisierung & Interpretation

Wir visualisieren das Verzeichnis und den Phasenplan, um den Ablauf greifbar zu machen.

In [2]:
# Wir prüfen die Existenz der Datendatei
data_file = 'testdata.txt'
if os.path.exists(data_file):
    file_size = os.path.getsize(data_file)
    print(f"Datensatz '{data_file}' gefunden. Dateigröße: {file_size} Bytes.")
else:
    print(f"WARNUNG: '{data_file}' nicht gefunden. Stellen Sie sicher, dass sich die Datei im selben Ordner befindet.")

Datensatz 'testdata.txt' gefunden. Dateigröße: 18194 Bytes.


--- 

## 7. Zwischenfazit

In dieser Phase 01 haben wir das biologische System verstanden (DSS-induzierte Colitis) und die gemessenen Variablen definiert. Wir haben die Notwendigkeit der mathematischen Normierung von `bwc` begründet und überprüft, ob unsere Python-Umgebung bereit ist. Der Grundstein für eine saubere, wissenschaftliche Analyse ist gelegt.

--- 

## 8. Quizfragen zur Selbstkontrolle

1. **Frage**: Warum normieren wir das Körpergewicht (`bwc`) auf den Tag 0, anstatt das absolute Gewicht in Gramm zu analysieren?
   * *Antwort*: Um biologisch bedingte Unterschiede im Startgewicht zwischen den Tieren auszugleichen und relative Änderungen direkt vergleichbar zu machen.
2. **Frage**: Zu welchem Typ von fehlenden Werten (*Missing Data Mechanism*) gehört das Fehlen von Datenpunkten durch Euthanasie bei Erreichen humaner Endpunkte?
   * *Antwort*: Es handelt sich um *Missing Not At Random (MNAR)*, da das Fehlen direkt vom (schlechten) Gesundheitszustand abhängt.
3. **Frage**: Welche Dosisgruppen von DSS werden in diesem Versuch verglichen?
   * *Antwort*: 0% (Kontrollgruppe), 1% (Niedrige Dosis) und 1.5% (Hohe Dosis).

--- 

## 9. Zusammenfassung & Hausaufgabe

### Zusammenfassung
- Wir untersuchen den Einfluss von DSS in drei Dosen (0%, 1%, 1.5%) auf Mäuse anhand von BWC (Körpergewicht) und VWR (Aktivität).
- Die Daten weisen durch Euthanasie ( humane Endpunkte) systemisch fehlende Werte auf (MNAR).

### Hausaufgabe
1. Machen Sie sich mit der Ordnerstruktur vertraut.
2. Prüfen Sie, ob Sie die Bibliotheken aus `requirements.txt` erfolgreich importieren können.

--- 

## 10. Weiterführende Literatur
- **Wirtz, S. et al. (2017)**: *Chemically induced mouse models of acute and chronic intestinal inflammation.* Nature Protocols, 12(7), 1295-1309. (Standardwerk zur Induktion von Colitis).
- **Tierschutzgesetz (TierSchG)**, Deutschland, Paragraph 7-9 (Grundlagen zum 3R-Prinzip).